# Tạo và Nạp dữ liệu đánh giá đơn vị kinh doanh UNIT_KPI (Problem Statement 5)
Notebook này nạp bảng kết quả đánh giá đơn vị kinh doanh (`unit_kpi.csv`) từ Problem Statement 5 vào Data Warehouse:
- **Nguồn dữ liệu:** `Problem/ProblemStatement5/unit_kpi.csv` (98 đơn vị: 43 Văn phòng + 55 Đại lý).
- **Mục tiêu:** Bảng `unit_kpi` phục vụ báo cáo phân tích hiệu quả kinh doanh, điểm KPI và đề xuất mở rộng trên Power BI.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse

# 1. Đọc dữ liệu từ thư mục ProblemStatement5
def find_file(filename):
    for prefix in ["../../Problem/ProblemStatement5/", "../Problem/ProblemStatement5/", "Problem/ProblemStatement5/"]:
        path = os.path.join(prefix, filename)
        if os.path.exists(path):
            return path
    return filename

csv_path = find_file("unit_kpi.csv")
df_kpi = pd.read_csv(csv_path)
print(f"Đã đọc {len(df_kpi)} dòng từ {csv_path}")

Đã đọc 98 dòng từ ../../Problem/ProblemStatement5/unit_kpi.csv


In [2]:
# 2. Chuẩn hóa và thêm Surrogate Key
df_kpi['unit_kpi_key'] = df_kpi.index + 1

# Sắp xếp lại thứ tự cột cho chuẩn
ordered_cols = [
    'unit_kpi_key', 'unit_id', 'unit_type', 'region', 'city', 'district',
    'total_customers', 'served_customers', 'invoice_count',
    'customer_percentile_in_city', 'customer_rank_in_city',
    'gross_revenue', 'refund_amount', 'net_revenue',
    'revenue_per_customer', 'revenue_per_invoice', 'customer_activation_pct',
    'net_revenue_score', 'served_customers_score', 'revenue_per_customer_score',
    'revenue_per_invoice_score', 'customer_activation_pct_score',
    'kpi_score', 'overall_rank', 'performance_group', 'recommendation'
]
df_kpi = df_kpi[ordered_cols]

print("Dữ liệu mẫu:")
display(df_kpi.head(3))
print(f"\nKích thước: {df_kpi.shape}")

Dữ liệu mẫu:


,unit_kpi_key,unit_id,unit_type,region,city,district,total_customers,served_customers,invoice_count,customer_percentile_in_city,...,customer_activation_pct,net_revenue_score,served_customers_score,revenue_per_customer_score,revenue_per_invoice_score,customer_activation_pct_score,kpi_score,overall_rank,performance_group,recommendation
0,1,Đại lý | East | Ha Long | District #03,Đại lý,East,Ha Long,District #03,344,287,1641,0.947368,...,83.430233,0.989796,0.928571,0.683673,0.857143,0.979592,88.775510,1,Hiệu quả cao,Ưu tiên mở rộng
1,2,Đại lý | East | Phu Ly | District #01,Đại lý,East,Phu Ly,District #01,355,255,1314,0.947368,...,71.830986,0.928571,0.795918,0.724490,1.000000,0.755102,84.081633,2,Hiệu quả cao,Ưu tiên mở rộng
2,3,Đại lý | East | Uong Bi | District #09,Đại lý,East,Uong Bi,District #09,297,235,1735,0.947368,...,79.124579,1.000000,0.658163,0.857143,0.591837,0.948980,81.122449,3,Hiệu quả cao,Ưu tiên mở rộng



Kích thước: (98, 26)


### Nạp dữ liệu vào PostgreSQL Data Warehouse

In [3]:
# 3. Kết nối PostgreSQL local
db_user = os.getenv("PGUSER", "postgres")
db_password = os.getenv("PGPASSWORD", "")
db_host = os.getenv("PGHOST", "localhost")
db_port = os.getenv("PGPORT", "5432")
db_name = os.getenv("PGDATABASE", "datawarehouse")
safe_password = urllib.parse.quote_plus(db_password)
DB_URL = f"postgresql://{db_user}:{safe_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(DB_URL)

# 4. Nạp vào bảng unit_kpi
df_kpi.to_sql('unit_kpi', con=engine, if_exists='replace', index=False)

print("\nĐã nạp thành công vào bảng UNIT_KPI trong Data Warehouse!")

# 5. Kiểm tra kiểm chứng lại số dòng đã nạp
check_count = pd.read_sql("SELECT COUNT(*) AS total_rows FROM unit_kpi", engine)
print(f"Số dòng trong DB: {check_count.iloc[0]['total_rows']}")

OperationalError: (psycopg.OperationalError) connection failed: connection to server at "127.0.0.1", port 5432 failed: fe_sendauth: no password supplied
Multiple connection attempts failed. All failures were:
- host: 'localhost', port: 5432, hostaddr: '::1': connection failed: connection to server at "::1", port 5432 failed: fe_sendauth: no password supplied
- host: 'localhost', port: 5432, hostaddr: '127.0.0.1': connection failed: connection to server at "127.0.0.1", port 5432 failed: fe_sendauth: no password supplied
(Background on this error at: https://sqlalche.me/e/21/e3q8)